<a href="https://colab.research.google.com/github/pskarthikk/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks, my rule, and its reason codes

### Signal checks

**1. Staleness — MIXED**

The declining rate rises from 51.14% for pages updated within 30 days to 61.11% for pages 91–180 days old. However, the 181+ bucket falls to 47.13% and contains only 174 pages. I therefore treat staleness as a directional signal rather than a proven predictor.

**2. Search visibility (`impressions_90d`) — MIXED**

The lowest-impression quartile has a 37.61% declining rate, while Q2 and Q3 are 60.46% and 62.56%. The highest quartile falls to 56.20%, so the relationship is not monotonic. I use impressions as an opportunity/visibility signal, not as a claim that high impressions alone cause decline.

### Baseline rule

Prioritize a page for refresh review when it has **at least 91 days since its last update** and **at least median 90-day search impressions (731)**.

The score combines a simple staleness level with existing search visibility. No fitted weights are used.

### Reason code

- `stale_and_visible` — page is at least 91 days old since its last update and has at least 731 impressions in the trailing 90 days.
- `not_priority` — page does not meet both baseline conditions.

### Action labels

- `refresh_review` — prioritize for editorial review.
- `monitor` — do not prioritize under this baseline.

In [3]:
!git clone https://github.com/pskarthikk/flyrank-ml-internship.git
%cd flyrank-ml-internship

!ls

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 137 (delta 49), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.87 MiB | 15.68 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/flyrank-ml-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [2]:
# Inspect the available files and folders needed for ML-07
from pathlib import Path

print("Current directory:", Path.cwd())
print("\nwork/notebooks exists:", Path("work/notebooks").exists())
print("data/processed exists:", Path("data/processed").exists())
print("data/raw exists:", Path("data/raw").exists())

if Path("data/processed").exists():
    print("\nProcessed files:")
    for p in Path("data/processed").iterdir():
        print(" -", p)

if Path("data/raw").exists():
    print("\nRaw files:")
    for p in Path("data/raw").iterdir():
        print(" -", p)

Current directory: /content

work/notebooks exists: False
data/processed exists: False
data/raw exists: False


In [4]:
from pathlib import Path

print("Available skills:")
for p in Path("skills").rglob("README.md"):
    print(p)

print("\nSkill directories:")
for p in Path("skills").iterdir():
    print("-", p)

Available skills:
skills/README.md

Skill directories:
- skills/building-baselines
- skills/deploying-static-pages
- skills/writing-data-contracts
- skills/README.md
- skills/writing-research-papers
- skills/flyrank
- skills/framing-ml-problems
- skills/hunting-leakage-and-validating
- skills/directing-your-ai-assistant
- skills/writing-honest-claims
- skills/querying-big-datasets
- skills/auditing-signals
- skills/training-honest-models


In [5]:
from pathlib import Path

files = [
    Path("skills/building-baselines/SKILL.md"),
    Path("skills/flyrank/flyrank-data/SKILL.md"),
]

for file in files:
    print("\n" + "=" * 80)
    print(file)
    print("=" * 80)
    print(file.read_text())


skills/building-baselines/SKILL.md
---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]     # readable on purpose
```

**3

In [6]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
for col in df.columns:
    print("-", col)

Rows: 30000
Columns: 44

Column names:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct


In [7]:
# Inspect the two candidate signals
signals = [
    "days_since_last_update",
    "impressions_90d"
]

print("Signal summary:\n")
print(df[signals].describe())

print("\nMissing values:\n")
print(df[signals].isna().sum())

print("\nStaleness values:\n")
print(df["days_since_last_update"].sort_values().head(10).to_list())
print("...")
print(df["days_since_last_update"].sort_values(ascending=False).head(10).to_list())

Signal summary:

       days_since_last_update  impressions_90d
count            30000.000000     30000.000000
mean                46.098300      5200.366300
std                 42.078709     16838.019547
min                  1.000000         1.000000
25%                 20.000000        81.000000
50%                 20.000000       731.000000
75%                104.000000      3615.250000
max                373.000000    517715.000000

Missing values:

days_since_last_update    0
impressions_90d           0
dtype: int64

Staleness values:

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
...
[373, 373, 373, 372, 372, 335, 334, 334, 313, 313]


In [8]:
print("trend_direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\ntrend_pct summary:")
print(df["trend_pct"].describe())

print("\nOther potentially useful categorical signals:")
print("\ncontent_type:")
print(df["content_type"].value_counts(dropna=False))

print("\nfreshness_tier:")
print(df["freshness_tier"].value_counts(dropna=False))

trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Other potentially useful categorical signals:

content_type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

freshness_tier:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64


In [9]:
# ML-07 Section 1 — Signal audit
# We use the observed "down" outcome ONLY to audit the signals.
# It will NOT be used as an input to the final scoring rule.

df["audit_declining"] = (df["trend_direction"] == "down").astype(int)

# ---------------------------------------------------------
# Signal 1: Staleness
# ---------------------------------------------------------
stale_bins = [-1, 30, 90, 180, float("inf")]
stale_labels = ["0-30", "31-90", "91-180", "181+"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels
)

stale_audit = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("audit_declining", "size"),
          declining_rate=("audit_declining", "mean")
      )
      .reset_index()
)

print("SIGNAL 1 — STALENESS")
print(stale_audit.to_string(index=False))
print()

# ---------------------------------------------------------
# Signal 2: Search visibility
# ---------------------------------------------------------
df["impressions_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"]
)

impressions_audit = (
    df.groupby("impressions_bucket", observed=False)
      .agg(
          n=("audit_declining", "size"),
          declining_rate=("audit_declining", "mean")
      )
      .reset_index()
)

print("SIGNAL 2 — IMPRESSIONS_90D")
print(impressions_audit.to_string(index=False))

SIGNAL 1 — STALENESS
staleness_bucket     n  declining_rate
            0-30 20480        0.511377
           31-90   175        0.588571
          91-180  9171        0.611057
            181+   174        0.471264

SIGNAL 2 — IMPRESSIONS_90D
impressions_bucket    n  declining_rate
         Q1_lowest 7503        0.376116
                Q2 7499        0.604614
                Q3 7498        0.625634
        Q4_highest 7500        0.562000


In [10]:
# Additional signal check: average search position

position_df = df[df["avg_position"] > 0].copy()

position_df["position_bucket"] = pd.qcut(
    position_df["avg_position"],
    q=4,
    labels=["Q1_best_position", "Q2", "Q3", "Q4_worst_position"]
)

position_audit = (
    position_df.groupby("position_bucket", observed=False)
    .agg(
        n=("audit_declining", "size"),
        declining_rate=("audit_declining", "mean")
    )
    .reset_index()
)

print("SIGNAL — AVG_POSITION")
print(position_audit.to_string(index=False))

SIGNAL — AVG_POSITION
  position_bucket    n  declining_rate
 Q1_best_position 7412        0.550594
               Q2 7076        0.583946
               Q3 7115        0.611806
Q4_worst_position 7192        0.512792


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)

print("Output directory ready:", Path("work/outputs").exists())

Output directory ready: True


In [14]:
# ML-07 Section 2 — Build the ranked baseline queue

# Thresholds chosen from the observed signal audit:
# - 91+ days since last update
# - median impressions_90d = 731

STALE_DAYS = 91
VISIBLE_IMPRESSIONS = 731

# Transparent rule components
df["stale_signal"] = (
    df["days_since_last_update"] >= STALE_DAYS
).astype(int)

df["visible_signal"] = (
    df["impressions_90d"] >= VISIBLE_IMPRESSIONS
).astype(int)

# Simple hand-written score.
# Visibility provides the ranking within the eligible refresh group.
df["score"] = (
    df["stale_signal"]
    * df["visible_signal"]
    * df["impressions_90d"]
)

# One reason code per page
df["reason_code"] = "not_priority"

df.loc[
    (df["stale_signal"] == 1) &
    (df["visible_signal"] == 1),
    "reason_code"
] = "stale_and_visible"

# Action label
df["action"] = "monitor"

df.loc[
    df["reason_code"] == "stale_and_visible",
    "action"
] = "refresh_review"

# Rank all pages by score, highest first
queue = (
    df[
        [
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
        ]
    ]
    .sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

# Put rank first
queue = queue[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
    ]
]

# Write the required output
output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Queue written to:", output_path)
print("Rows:", len(queue))
print("\nAction counts:")
print(queue["action"].value_counts())

print("\nTop 10:")
print(queue.head(10).to_string(index=False))

Queue written to: work/outputs/baseline_action_score.csv
Rows: 30000

Action counts:
action
monitor           24008
refresh_review     5992
Name: count, dtype: int64

Top 10:
 rank           content_id  score       reason_code         action  days_since_last_update  impressions_90d
    1 content_5fe46e04994d 517715 stale_and_visible refresh_review                     104           517715
    2 content_2dba2b1f9536 443434 stale_and_visible refresh_review                     104           443434
    3 content_2c2606c5d176 347399 stale_and_visible refresh_review                     104           347399
    4 content_cb112fce36be 309910 stale_and_visible refresh_review                     104           309910
    5 content_9532f197bbc8 309192 stale_and_visible refresh_review                     104           309192
    6 content_36ff89c8214e 295097 stale_and_visible refresh_review                     104           295097
    7 content_b28d1efd668f 286608 stale_and_visible refresh_review   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
# ML-07 Section 3 — Top-20 manual review

top20_review = [
    {
        "rank": 1,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High visibility and 104 days since update make this a strong baseline candidate.",
        "what_would_make_it_wrong": "The page may already be accurate and current despite the age, or the impressions may not represent useful current opportunity."
    },
    {
        "rank": 2,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "Very high visibility and 104 days since update support review priority.",
        "what_would_make_it_wrong": "A refresh may not improve performance if the underlying content is already satisfying the search intent."
    },
    {
        "rank": 3,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High visibility, strong clicks, and position 4.2 suggest meaningful existing search value.",
        "what_would_make_it_wrong": "The page may already be performing well enough that changing it creates more risk than opportunity."
    },
    {
        "rank": 4,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and 104 days since update make it a reasonable refresh candidate.",
        "what_would_make_it_wrong": "The page could be intentionally stable and need no substantive update."
    },
    {
        "rank": 5,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "Very high impressions and strong click volume indicate substantial existing visibility.",
        "what_would_make_it_wrong": "Its position 2.0 and 0.87% CTR may indicate that a refresh has limited upside."
    },
    {
        "rank": 6,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "Very high impressions combined with a low 0.05% CTR makes review potentially useful.",
        "what_would_make_it_wrong": "The low CTR may be explained by query mix or measurement context rather than stale content."
    },
    {
        "rank": 7,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High visibility and 104 days since update make this a plausible opportunity.",
        "what_would_make_it_wrong": "Average position 26.2 may indicate that the page needs a broader SEO intervention rather than a simple content refresh."
    },
    {
        "rank": 8,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and substantial sessions provide evidence of existing audience reach.",
        "what_would_make_it_wrong": "Position 26.2 may mean the limiting factor is ranking rather than content freshness."
    },
    {
        "rank": 9,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions, clicks, and position 5.1 indicate a page with meaningful search value.",
        "what_would_make_it_wrong": "The page may already be performing strongly enough that a refresh would add little value."
    },
    {
        "rank": 10,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "Very high impressions make it visible to the baseline, but zero clicks makes this recommendation suspicious.",
        "what_would_make_it_wrong": "The impression data may reflect low-value queries or a mismatch between visibility and genuine user interest."
    },
    {
        "rank": 11,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and 937 sessions provide evidence of meaningful existing traffic.",
        "what_would_make_it_wrong": "A refresh may not improve results if the content already matches user needs."
    },
    {
        "rank": 12,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High visibility with position 5.8 gives the page a plausible refresh opportunity.",
        "what_would_make_it_wrong": "Strong ranking could mean there is little incremental benefit from changing the page."
    },
    {
        "rank": 13,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and position 5.7 provide meaningful existing search visibility.",
        "what_would_make_it_wrong": "Low click volume relative to impressions may indicate that freshness is not the main problem."
    },
    {
        "rank": 14,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and 1,024 sessions indicate meaningful audience exposure.",
        "what_would_make_it_wrong": "Position 12.5 may indicate a ranking problem that content refresh alone may not solve."
    },
    {
        "rank": 15,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and position 4.3 make this a credible high-value review candidate.",
        "what_would_make_it_wrong": "The page may already be near its practical performance ceiling."
    },
    {
        "rank": 16,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions, strong clicks, and position 4.0 suggest substantial existing value.",
        "what_would_make_it_wrong": "A refresh could disrupt a page that is already performing well."
    },
    {
        "rank": 17,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High visibility and 104 days since update support review under the baseline.",
        "what_would_make_it_wrong": "Low CTR relative to its visibility may require diagnosing search intent before refreshing."
    },
    {
        "rank": 18,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions combined with position 25.8 creates a plausible opportunity for investigation.",
        "what_would_make_it_wrong": "The weak position may indicate a relevance or authority problem rather than staleness."
    },
    {
        "rank": 19,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions and 549 sessions show meaningful existing reach.",
        "what_would_make_it_wrong": "Position 22.1 and low CTR may indicate that the issue is not simply content age."
    },
    {
        "rank": 20,
        "action": "refresh_review",
        "reason_code": "stale_and_visible",
        "confidence_note": "High impressions, 536 sessions, and position 4.3 support review priority.",
        "what_would_make_it_wrong": "The page already ranks strongly, so a refresh may have limited incremental benefit."
    },
]

review_df = pd.DataFrame(top20_review)

print("TOP-20 MANUAL REVIEW")
print(review_df.to_string(index=False))

TOP-20 MANUAL REVIEW
 rank         action       reason_code                                                                                              confidence_note                                                                                                       what_would_make_it_wrong
    1 refresh_review stale_and_visible                             High visibility and 104 days since update make this a strong baseline candidate. The page may already be accurate and current despite the age, or the impressions may not represent useful current opportunity.
    2 refresh_review stale_and_visible                                      Very high visibility and 104 days since update support review priority.                       A refresh may not improve performance if the underlying content is already satisfying the search intent.
    3 refresh_review stale_and_visible                   High visibility, strong clicks, and position 4.2 suggest meaningful existing search value.       

In [15]:
# ML-07 Section 3 — Inspect the Top 20 for manual review

review_cols = [
    "content_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_type",
    "content_age_days",
]

top20_detail = (
    df.sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )
    .head(20)[review_cols]
    .reset_index(drop=True)
)

top20_detail.insert(0, "rank", range(1, 21))

print(top20_detail.to_string(index=False))

 rank           content_id  score       reason_code         action  days_since_last_update  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr    content_type  content_age_days
    1 content_5fe46e04994d 517715 stale_and_visible refresh_review                     104           517715         741           520           4.2 0.14 keyword article               537
    2 content_2dba2b1f9536 443434 stale_and_visible refresh_review                     104           443434         910          4218          27.9 0.21 keyword article               299
    3 content_2c2606c5d176 347399 stale_and_visible refresh_review                     104           347399        1854          2146           4.2 0.53 keyword article               362
    4 content_cb112fce36be 309910 stale_and_visible refresh_review                     104           309910         492           480           5.6 0.16 keyword article               126
    5 content_9532f197bbc8 309192 stale_and_visible refresh_revie

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.